In [1]:

from dotenv import load_dotenv


load_dotenv()

True

In [4]:
import asyncio
import httpx

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}


class RetryMCPInterceptor:
    def __init__(self, max_retries=3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None

        for attempt in range(self.max_retries):
            try:
                return await handler(request)

            except McpError as exc:
                last_error = exc

                print(
                    f"[MCP] Retry {attempt + 1}/{self.max_retries} "
                    f"Code={exc.error.code}"
                )

                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[
                            TextContent(
                                type="text",
                                text=f"Non Retryable Error: {exc}"
                            )
                        ],
                        isError=False,
                    )

            except Exception as exc:
                last_error = exc
                print(f"[MCP] Retry {attempt + 1}/{self.max_retries}")
                print(exc)

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        return CallToolResult(
            content=[
                TextContent(
                    type="text",
                    text=f"Failed after retries: {last_error}"
                )
            ],
            isError=False,
        )


async def connect_to_server():

    for attempt in range(3):

        try:

            client = MultiServerMCPClient(
                {
                    "travel_server": {
                        "transport": "streamable_http",
                        "url": "https://mcp.kiwi.com"
                    }
                },
                tool_interceptors=[
                    RetryMCPInterceptor()
                ],
            )

            tools = await client.get_tools()

            print("Connected Successfully")
            print(tools)

            return tools

        except httpx.ConnectError as e:

            print(f"Connection Retry {attempt + 1}/3")
            print(e)

        except Exception as e:

            print(f"Unexpected Retry {attempt + 1}/3")
            print(type(e).__name__)
            print(e)

        await asyncio.sleep(2)

    print("Unable to connect to MCP Server.")
    return []


# Jupyter Notebook
tools = await connect_to_server()

print("Loaded tools:", tools)
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

Unexpected Retry 1/3
ExceptionGroup
unhandled errors in a TaskGroup (1 sub-exception)
Unexpected Retry 2/3
ExceptionGroup
unhandled errors in a TaskGroup (1 sub-exception)
Unexpected Retry 3/3
ExceptionGroup
unhandled errors in a TaskGroup (1 sub-exception)
Unable to connect to MCP Server.
Loaded tools: []


  + Exception Group Traceback (most recent call last):
  |   File "E:\vs-langchain-agents\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\Admin new\AppData\Local\Temp\ipykernel_22276\3412352033.py", line 163, in <module>
  |     tools = await client.get_tools()
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "E:\vs-langchain-agents\.venv\Lib\site-packages\langchain_mcp_adapters\client.py", line 209, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "E:\vs-langchain-agents\.venv\Lib\site-packages\langchain_mcp_adapters\tools.py", line 590, in load_mcp_tools
  |     async with create_session(
  |                ^^^^^^^^^^^^^^^
  |   File "C:\Users\Admin new\AppData\Local\Programs\Python\Python312\Lib\contextlib.py", line 231, in __aexit__
  |     await self.gen.athro

In [5]:
from typing import Dict,Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client=TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [6]:
from langchain_core.tools import tool
from langchain_community.utilities import SQLDatabase

sql_db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information."""
    try:
        return sql_db.run(query)
    except Exception as e:
        return f"Error querying database: {str(e)}"

C:\Users\Admin new\AppData\Local\Temp\ipykernel_22276\3609452126.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [7]:
from langchain.agents import AgentState
class WeddingState(AgentState):
    origin:str
    destination:str
    guest_count:str
    genre:str



In [8]:
from langchain.agents import create_agent

# Travel agent
travel_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [9]:
# Venue agent
venue_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [10]:
# Playlist agent
playlist_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

In [18]:
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langchain.tools import  ToolRuntime
from langchain_core.tools import tool
from langchain.messages import HumanMessage

@tool
async def search_flights(origin: str, destination: str) -> str:
    """Search for flights."""
    try:
        response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find the cheapest one-way economy flight from {origin} to {destination}.")]})
        return response["messages"][-1].content
    except Exception as e:
        return f"Flight search failed: {str(e)}"

@tool
def search_venues(location: str, capacity: int) -> str:
    """Search for wedding venues."""
    try:
        response = venue_agent.invoke({"messages": [HumanMessage(content=f"Find the best wedding venues in {location} for {capacity} guests.")]})
        return response["messages"][-1].content
    except Exception as e:
        return f"Venue search failed: {str(e)}"

@tool
def suggest_playlist(genre: str) -> str:
    """Suggest a wedding playlist."""
    try:
        response = playlist_agent.invoke({"messages": [HumanMessage(content=f"Create a wedding playlist with {genre} songs.")]})
        return response["messages"][-1].content
    except Exception as e:
        return f"Playlist generation failed: {str(e)}"

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre.
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [19]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator.
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


In [20]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool search_venues did not match schema: errors: [`/capacity`: expected integer, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_venues>{"capacity": "100", "location": "Paris"}</function>'}}

In [21]:
from pprint import pprint
pprint(response)

NameError: name 'response' is not defined

In [26]:
# Wedding planner multi-agent system
#
# Fixes vs. the original:
# - Removed the duplicated MCP interceptor / connection block (it was defined
#   twice; the second copy silently dropped the retry wrapper).
# - Sub-agent tools (`search_venues`, `suggest_playlist`) are now `async def`
#   using `await agent.ainvoke(...)` instead of blocking `.invoke()`, matching
#   `search_flights`.
# - `update_state` is now correctly annotated as returning a `Command`.
# - `guest_count` is `int` end-to-end instead of `str`.
# - Agents are (re)built after the MCP tools are confirmed connected.

from dotenv import load_dotenv

load_dotenv()

import asyncio
import httpx

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}  # internal error -> worth retrying


class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(
                    f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                    f"(code {exc.error.code}, attempt {attempt + 1}/{self.max_retries}): {exc}"
                )
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(
                    f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                    f"(attempt {attempt + 1}/{self.max_retries}): {exc}"
                )

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[
                TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )


def _print_exception_chain(exc: BaseException, indent: str = ""):
    """anyio/httpx wrap real errors inside ExceptionGroup, which hides the
    actual cause behind a generic 'TaskGroup' message. Unwrap it recursively
    so you can see what actually failed (SSL error, 404, timeout, DNS, etc.)."""
    if isinstance(exc, BaseExceptionGroup):
        print(f"{indent}{type(exc).__name__}: {exc.message}")
        for sub in exc.exceptions:
            _print_exception_chain(sub, indent + "  ")
    else:
        print(f"{indent}{type(exc).__name__}: {exc}")
        if exc.__cause__:
            print(f"{indent}caused by:")
            _print_exception_chain(exc.__cause__, indent + "  ")


async def connect_to_server():
    """Connect to the Kiwi travel MCP server, retrying the connection itself
    (separate from per-tool-call retries handled by RetryMCPInterceptor)."""
    for attempt in range(3):
        try:
            client = MultiServerMCPClient(
                {
                    "travel_server": {
                        "transport": "streamable_http",
                        "url": "https://mcp.tavily.com/mcp/?tavilyApiKey=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",
                    }
                },
                tool_interceptors=[RetryMCPInterceptor()],
            )
            tools = await client.get_tools()
            print("Connected Successfully")
            print(tools)
            return tools

        except httpx.ConnectError as e:
            print(f"Connection Retry {attempt + 1}/3")
            print(e)

        except Exception as e:
            print(f"Unexpected Retry {attempt + 1}/3")
            _print_exception_chain(e)  # <-- shows the real error, not just "ExceptionGroup"

        await asyncio.sleep(2)

    print("Unable to connect to MCP Server.")
    return []


# Jupyter notebook: top-level await is fine here
tools = await connect_to_server()
print("Loaded tools:", tools)

from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()


@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters
    (e.g., use 'capacite' instead of 'capacite').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}


from langchain_core.tools import tool
from langchain_community.utilities import SQLDatabase

sql_db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information."""
    try:
        return sql_db.run(query)
    except Exception as e:
        return f"Error querying database: {str(e)}"


from langchain.agents import AgentState


class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: int
    genre: str


from langchain.agents import create_agent

# Travel agent
travel_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

# Venue agent
venue_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

# Playlist agent
playlist_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langchain.tools import ToolRuntime
from langchain_core.tools import tool
from langchain.messages import HumanMessage


@tool
async def search_flights(origin: str, destination: str) -> str:
    """Search for flights."""
    try:
        response = await travel_agent.ainvoke(
            {"messages": [
                HumanMessage(content=f"Find the cheapest one-way economy flight from {origin} to {destination}.")]}
        )
        return response["messages"][-1].content
    except Exception as e:
        return f"Flight search failed: {str(e)}"


@tool
async def search_venues(location: str, capacity: int) -> str:
    """Search for wedding venues."""
    try:
        response = await venue_agent.ainvoke(
            {"messages": [HumanMessage(content=f"Find the best wedding venues in {location} for {capacity} guests.")]}
        )
        return response["messages"][-1].content
    except Exception as e:
        return f"Venue search failed: {str(e)}"


@tool
async def suggest_playlist(genre: str) -> str:
    """Suggest a wedding playlist."""
    try:
        response = await playlist_agent.ainvoke(
            {"messages": [HumanMessage(content=f"Create a wedding playlist with {genre} songs.")]}
        )
        return response["messages"][-1].content
    except Exception as e:
        return f"Playlist generation failed: {str(e)}"


@tool
def update_state(
        origin: str, destination: str, guest_count: int, genre: str, runtime: ToolRuntime
) -> Command:
    """Update the state when you know all of the values: origin, destination, guest_count, genre.
    This tool must be called alone, without any other tool calls. It must complete and return to make
    the information available to other tools."""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
    )


from langchain.agents import create_agent

coordinator = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator.
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)

from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [
            HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  # tag traces for Langsmith; allow up to 40 steps
)

from pprint import pprint

pprint(response)

Connected Successfully
[StructuredTool(name='tavily_search', description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', args_schema={'additionalProperties': False, 'properties': {'query': {'description': 'Search query', 'type': 'string'}, 'max_results': {'default': 5, 'description': 'The maximum number of search results to return', 'type': 'integer'}, 'search_depth': {'default': 'basic', 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'type': 'string'}, 'topic': {'const': 'general', 'default': 'general', 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'type': 'string'}, 'time_range': {'anyOf': [{'enum': 

BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool search_venues did not match schema: errors: [`/capacity`: expected integer, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_venues>{"capacity": "100", "location": "Paris"}</function>\n'}}